<a href="https://colab.research.google.com/github/SSatish178/CloudDevopsPracitice/blob/main/satish_day2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain-groq langgraph langchain-core

In [2]:
import os, getpass
os.environ["GROQ_API_KEY"] = getpass.getpass("Paste your Groq API key: ")

Paste your Groq API key: ··········


In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=3)

print(llm.invoke("What is the price of a chocolate cake at my bakery? Is it open on Monday?").content)

NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [4]:
import requests, os
r = requests.get("https://api.groq.com/openai/v1/models",
                 headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"})
for m in r.json()["data"]:
    print(m["id"])

canopylabs/orpheus-arabic-saudi
whisper-large-v3-turbo
canopylabs/orpheus-v1-english
qwen/qwen3.8-27b
allam-2-7b
openai/gpt-oss-20b
whisper-large-v3
openai/gpt-oss-120b
meta-llama/llama-prompt-guard-2-22m
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-safeguard-20b


In [5]:
from langchain_groq import ChatGroq

MODEL = "openai/gpt-oss-120b"   # change this if your list shows a different name
llm = ChatGroq(model=MODEL, temperature=0, max_retries=3)

print(llm.invoke("What is the price of a chocolate cake at my bakery? Is it open on Monday?").content)

I’m sorry, but I don’t have any information about your bakery’s menu or hours. If you can share the details you’re looking for (e.g., the name of the bakery, its location, or any publicly available website), I’ll do my best to help you find the answer.


In [8]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# 1. మెను మరియు ధరలు
PRICES = {"chocolate cake": 650, "plum cake": 500, "cupcake": 80}

# 2. టూల్ 1: ధర కనుగొనడానికి
@tool
def get_price(item: str) -> str:
    """Return the price in rupees of one bakery item, e.g. 'plum cake'."""
    price = PRICES.get(item.lower())
    return f"Rs {price}" if price else "Not on the menu"

# 3. టూల్ 2: బేకరీ తెరిచి ఉందా లేదా చెక్ చేయడానికి
@tool
def is_open(day: str) -> str:
    """Say whether the bakery takes orders on a given weekday."""
    return "Closed" if day.lower() == "monday" else "Open"

# 4. నువ్వు సక్సెస్ అయిన మోడల్ పేరుతో LLM ని సెటప్ చేయడం
MODEL = "openai/gpt-oss-120b"
llm = ChatGroq(model=MODEL, temperature=0, max_retries=3)

# 5. టూల్స్‌ని మరియు LLM ని కలిపి ఏజెంట్ తయారు చేయడం
agent = create_react_agent(llm, [get_price, is_open])

# 6. కస్టమర్ అడిగే ప్రశ్నను ఏజెంట్‌కి ఇవ్వడం
result = agent.invoke({"messages": [("human", "Can I order a chocolate cake for Sunday? Price?")]})

# 7. ఏజెంట్ స్టెప్-బై-స్టెప్ ఎలా ఆలోచించిందో ప్రింట్ చేయడం
for m in result["messages"]:
    m.pretty_print()


/tmp/ipykernel_556/477990357.py:26: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, [get_price, is_open])


================================ Human Message =================================

Can I order a chocolate cake for Sunday? Price?
================================== Ai Message ==================================
Tool Calls:
  is_open (fc_1bd906fc-f12d-404e-ba1e-017d815122e6)
 Call ID: fc_1bd906fc-f12d-404e-ba1e-017d815122e6
  Args:
    day: Sunday
================================= Tool Message =================================
Name: is_open

Open
================================== Ai Message ==================================
Tool Calls:
  get_price (fc_8ebb1544-9798-4dcf-a6dc-42ec8bdf9267)
 Call ID: fc_8ebb1544-9798-4dcf-a6dc-42ec8bdf9267
  Args:
    item: chocolate cake
================================= Tool Message =================================
Name: get_price

Rs 650
================================== Ai Message ==================================

Yes, you can place an order for a chocolate cake on Sunday—the bakery is open that day.  
The price for a chocolate cake is **₹ 650**

In [6]:
from langchain_core.tools import tool

# Our bakery menu and prices
PRICES = {"chocolate cake": 650, "plum cake": 500, "cupcake": 80}

@tool
def get_price(item: str) -> str:
    """Return the price in rupees of one bakery item, e.g. 'plum cake'."""
    price = PRICES.get(item.lower())
    return f"Rs {price}" if price else "Not on the menu"

@tool
def is_open(day: str) -> str:
    """Say whether the bakery takes orders on a given weekday."""
    return "Closed" if day.lower() == "monday" else "Open"

print(get_price.invoke("plum cake"))
print(is_open.invoke("Monday"))

Rs 500
Closed


In [9]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

PRICES = {"chocolate cake": 650, "plum cake": 500, "cupcake": 80}

@tool
def get_price(item: str) -> str:
    """Return the price in rupees of one bakery item like 'chocolate cake' or 'plum cake'."""
    price = PRICES.get(item.lower())
    return f"Rs {price}" if price else "Not on the menu"

@tool
def is_open(day: str) -> str:
    """Check if bakery is open on a specific day of the week (e.g. 'Monday', 'Sunday')."""
    valid_days = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
    d = day.lower().strip()
    if d not in valid_days:
        return "Please specify a valid weekday (e.g., Monday, Tuesday)."
    return "Closed" if d == "monday" else "Open"

# 1. LLM Setup
MODEL = "openai/gpt-oss-120b"
llm = ChatGroq(model=MODEL, temperature=0, max_retries=3)

# 2. Agent Setup
agent = create_react_agent(llm, [get_price, is_open])

# 3. Test Question
result = agent.invoke({"messages": [("human", "Can I get a plum cake on Sunday? Price?")]})

for m in result["messages"]:
    m.pretty_print()

/tmp/ipykernel_556/2717258990.py:27: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, [get_price, is_open])


================================ Human Message =================================

Can I get a plum cake on Sunday? Price?
================================== Ai Message ==================================
Tool Calls:
  is_open (fc_d4b1bbc4-9f17-4ee7-bbc1-77dfdfc0600e)
 Call ID: fc_d4b1bbc4-9f17-4ee7-bbc1-77dfdfc0600e
  Args:
    day: Sunday
================================= Tool Message =================================
Name: is_open

Open
================================== Ai Message ==================================
Tool Calls:
  get_price (fc_48f4a1cb-2d0c-439d-9b50-a65f3892d4a9)
 Call ID: fc_48f4a1cb-2d0c-439d-9b50-a65f3892d4a9
  Args:
    item: plum cake
================================= Tool Message =================================
Name: get_price

Rs 500
================================== Ai Message ==================================

Yes! The bakery is open on Sundays, and a plum cake costs **₹ 500**. Let me know if you’d like to place an order or need any other details.


In [7]:
from langchain_groq import ChatGroq

# మోడల్ పేరు మార్చి రన్ చేయి
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=3)

print(llm.invoke("What is the price of a chocolate cake at my bakery? Is it open on Monday?").content)


NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [10]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# 1. మెను మరియు ధరలు
PRICES = {"chocolate cake": 650, "plum cake": 500, "cupcake": 80}

# 2. టూల్ 1: ధర కనుగొనడానికి
@tool
def get_price(item: str) -> str:
    """Return the price in rupees of one bakery item, e.g. 'plum cake'."""
    price = PRICES.get(item.lower())
    return f"Rs {price}" if price else "Not on the menu"

# 3. టూల్ 2: బేకరీ తెరిచి ఉందా లేదా చెక్ చేయడానికి
@tool
def is_open(day: str) -> str:
    """Say whether the bakery takes orders on a given weekday."""
    return "Closed" if day.lower() == "monday" else "Open"

# 4. నువ్వు సక్సెస్ అయిన మోడల్ పేరుతో LLM ని సెటప్ చేయడం
MODEL = "openai/gpt-oss-120b"
llm = ChatGroq(model=MODEL, temperature=0, max_retries=3)

# 5. టూల్స్‌ని మరియు LLM ని కలిపి ఏజెంట్ తయారు చేయడం
agent = create_react_agent(llm, [get_price, is_open])

# 6. కస్టమర్ అడిగే ప్రశ్నను ఏజెంట్‌కి ఇవ్వడం
result = agent.invoke({"messages": [("human", "Can I order a chocolate cake for Sunday? Price?")]})

# 7. ఏజెంట్ స్టెప్-బై-స్టెప్ ఎలా ఆలోచించిందో ప్రింట్ చేయడం
for m in result["messages"]:
    m.pretty_print()

/tmp/ipykernel_556/3773934672.py:26: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, [get_price, is_open])


================================ Human Message =================================

Can I order a chocolate cake for Sunday? Price?
================================== Ai Message ==================================
Tool Calls:
  is_open (fc_8887d72d-2351-46fe-9ab1-e5ce30f46858)
 Call ID: fc_8887d72d-2351-46fe-9ab1-e5ce30f46858
  Args:
    day: Sunday
================================= Tool Message =================================
Name: is_open

Open
================================== Ai Message ==================================
Tool Calls:
  get_price (fc_d6dd106c-45f4-4b82-bf38-ca831710f50a)
 Call ID: fc_d6dd106c-45f4-4b82-bf38-ca831710f50a
  Args:
    item: chocolate cake
================================= Tool Message =================================
Name: get_price

Rs 650
================================== Ai Message ==================================

Yes, you can place an order for a chocolate cake on Sunday—the bakery is open that day.  
The price for a chocolate cake is **₹ 650**

In [11]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
PRICES = {"chocolate cake": 650, "plum cake": 500, "cupcake": 80}
@tool
def get_price(item: str) -> str:
"""Return the price in rupees of one bakery item, e.g. 'plum cake'."""
price = PRICES.get(item.lower())
return f"Rs {price}" if price else "Not on the menu"
@tool
def is_open(day: str) -> str:
"""Say whether the bakery takes orders on a given weekday."""
return "Closed" if day.lower() == "monday" else "Open"
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0, max_retries=3)
agent = create_react_agent(llm, [get_price, is_open])
result = agent.invoke({"messages": [("human", "Can I order a chocolate cake for Sunday? Price?")]})
for m in result["messages"]: # shows each tool call step by step
m.pretty_print()

IndentationError: expected an indented block after function definition on line 6 (1395622204.py, line 7)

In [ ]:
from langchain_core.tools import tool

# మన బేకరీ మెను మరియు ధరలు
PRICES = {"chocolate cake": 650, "plum cake": 500, "cupcake": 80}

@tool
def get_price(item: str) -> str:
    """Return the price in rupees of one bakery item, e.g. 'plum cake'."""
    price = PRICES.get(item.lower())
    return f"Rs {price}" if price else "Not on the menu"

@tool
def is_open(day: str) -> str:
    """Say whether the bakery takes orders on a given weekday."""
    return "Closed" if day.lower() == "monday" else "Open"

print(get_price.invoke("plum cake"))
print(is_open.invoke("Monday"))

In [12]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

PRICES = {"chocolate cake": 650, "plum cake": 500, "cupcake": 80}

@tool
def get_price(item: str) -> str:
    """Return the price in rupees of one bakery item, e.g. 'plum cake'."""
    price = PRICES.get(item.lower())
    return f"Rs {price}" if price else "Not on the menu"

@tool
def is_open(day: str) -> str:
    """Say whether the bakery takes orders on a given weekday."""
    return "Closed" if day.lower() == "monday" else "Open"

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0, max_retries=3)

agent = create_react_agent(llm, [get_price, is_open])

result = agent.invoke({"messages": [("human", "Can I order a chocolate cake for Sunday? Price?")]})

for m in result["messages"]:  # shows each tool call step by step
    m.pretty_print()

/tmp/ipykernel_556/1717415953.py:20: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, [get_price, is_open])


================================ Human Message =================================

Can I order a chocolate cake for Sunday? Price?
================================== Ai Message ==================================
Tool Calls:
  is_open (fc_5390013e-64cd-4923-a3bc-5f9591159ea4)
 Call ID: fc_5390013e-64cd-4923-a3bc-5f9591159ea4
  Args:
    day: Sunday
================================= Tool Message =================================
Name: is_open

Open
================================== Ai Message ==================================
Tool Calls:
  get_price (fc_2f1d3586-0baf-4cec-a9bc-c7e5094c2e54)
 Call ID: fc_2f1d3586-0baf-4cec-a9bc-c7e5094c2e54
  Args:
    item: chocolate cake
================================= Tool Message =================================
Name: get_price

Rs 650
================================== Ai Message ==================================

Yes, you can place an order for a chocolate cake on Sunday—the bakery is open that day.  
The price for a chocolate cake is **₹ 650**